# Grilla de imágenes NOGSE: CPMG, Hahn y CPMG − Hahn vs $G$

Una fila por serie (por defecto CPMG y Hahn del mismo $T_{NOGSE}$) y una columna por gradiente $G$; la última fila es la diferencia píxel a píxel CPMG − Hahn. Lee directamente los NIfTI 4D con sus `.bval`/`.bvec`, así que sirve para cualquier par de series o para la rama `raw` o `den_gr` cambiando solo la primera celda.

- **Columnas.** $b\propto G^2$ para una temporización fija, así que $G = G_{max}\sqrt{b/b_{max}}$ sale del `.bval` sin usar ningún modelo de forma de onda. La columna $G=0$ es la media de los volúmenes b0.
- **Dirección.** Cada $G>0$ tiene tres volúmenes (bvec $x$, $y$, $z$): se elige una dirección o la media de las tres.
- **Escala.** Las filas de señal comparten la escala de grises (0 → percentil `VMAX_PERCENTILE` de los b0). La diferencia usa una escala simétrica alrededor de 0 (gris = 0, blanco = CPMG > Hahn).

Series de `sub-PHANTOM-1`: s3/s4 ($T$ = 30 ms), s6/s7 (40 ms), s9/s10 (50 ms), s12/s13 (70 ms), siempre CPMG/Hahn.

In [ ]:
from pathlib import Path

PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'Data-BIDS').is_dir())

# Rama denoised + Gibbs (img_preprocessing). Para la rama raw, descomentar las dos líneas siguientes.
DWI_DIR = PROJECT_ROOT / 'Data-BIDS' / 'derivatives' / 'img_preprocessing' / 'sub-PHANTOM-1' / 'ses-T0' / 'dwi'
SUFFIX = '_desc-denGr_dwi.nii.gz'
# DWI_DIR = PROJECT_ROOT / 'Data-BIDS' / 'sub-PHANTOM-1' / 'ses-T0' / 'dwi'
# SUFFIX = '_dwi.nii.gz'

# Una fila por serie: {etiqueta de la fila: NIfTI 4D}. Los .bval/.bvec/.json se leen al lado del NIfTI.
ROWS = {
    'CPMG': DWI_DIR / f'sub-PHANTOM-1_ses-T0_acq-s3{SUFFIX}',
    'HAHN': DWI_DIR / f'sub-PHANTOM-1_ses-T0_acq-s4{SUFFIX}',
}
DIFFERENCE = ('CPMG', 'HAHN')  # fila extra ROWS[a] - ROWS[b]; None para no dibujarla

OUTPUT_DIR = PROJECT_ROOT / 'repos' / 'signal_analysis' / 'fittings' / 'image_grids'
OUTPUT_NAME = None  # None: nombre automático a partir de las series y la dirección

## Opciones de visualización

In [ ]:
G_MAX_MT_M = 70.0          # G del volumen de b máximo; G = G_MAX * sqrt(b / b_max)
DIRECTION = 'mean'         # 1, 2, 3 (eje x, y, z del bvec) o 'mean' (media de las tres)
SLICE = 0
NORMALIZE = False          # True: S/S0 por píxel (S0 = media de los b0 de la misma serie)
BACKGROUND_FRACTION = 0.05 # con NORMALIZE, los píxeles con S0 < fracción * p99(S0) quedan en 0
VMAX_PERCENTILE = 99.5
DIFF_LIMIT = None          # límite simétrico de la fila de diferencia; None = percentil VMAX_PERCENTILE de |diferencia|
ROTATE_K = 1               # np.rot90(imagen, ROTATE_K): 1 = anterior arriba
FLIP_LR = False
CROP = None                # (fila0, fila1, col0, col1) sobre la imagen ya orientada, o None
TITLE = None               # None: protocolos de las series + dirección

In [ ]:
import json

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from IPython.display import display


def sidecar(path, ext):
    for suffix in ('.nii.gz', '.nii'):
        if path.name.endswith(suffix):
            return path.with_name(path.name[:-len(suffix)] + ext)
    raise ValueError(f'{path} no es un NIfTI.')


def load_series(path):
    path = Path(path)
    data = np.asarray(nib.load(path).dataobj, dtype=float)
    if data.ndim != 4:
        raise ValueError(f'{path.name}: se espera un NIfTI 4D y la forma es {data.shape}.')
    data = data[:, :, SLICE, :]
    bval = np.loadtxt(sidecar(path, '.bval')).ravel()
    bvec = np.loadtxt(sidecar(path, '.bvec')).reshape(3, -1)
    if bval.size != data.shape[-1] or bvec.shape[1] != data.shape[-1]:
        raise ValueError(f'{path.name}: {data.shape[-1]} volúmenes, {bval.size} bvals y {bvec.shape[1]} bvecs.')
    g = np.round(G_MAX_MT_M * np.sqrt(bval / bval.max()), 2)
    direction = np.where(g > 0, np.argmax(np.abs(bvec), axis=0) + 1, 0)
    meta_path = sidecar(path, '.json')
    meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}
    protocol = Path(meta.get('WipMemBlock', meta.get('ProtocolName', path.name))).stem
    return {'path': path, 'data': data, 'bval': bval, 'g': g, 'direction': direction, 'protocol': protocol}


def g_stack(series):
    # (n_G, X, Y): media de los b0 en G=0 y, para G>0, la dirección elegida o la media de las tres
    levels = np.unique(series['g'])
    images = []
    for g in levels:
        selected = series['g'] == g
        if g > 0 and DIRECTION != 'mean':
            selected &= series['direction'] == int(DIRECTION)
        if not selected.any():
            raise ValueError(f'{series["path"].name}: no hay volúmenes con G={g:g} mT/m y dirección {DIRECTION}.')
        images.append(series['data'][..., selected].mean(axis=-1))
    images = np.stack(images)
    if NORMALIZE:
        s0 = images[0]
        keep = s0 > BACKGROUND_FRACTION * np.percentile(s0, 99)
        images = np.where(keep, images / np.where(keep, s0, 1.0), 0.0)
    return levels, images


def orient(image):
    image = np.rot90(image, ROTATE_K)
    if FLIP_LR:
        image = image[:, ::-1]
    if CROP is not None:
        r0, r1, c0, c1 = CROP
        image = image[r0:r1, c0:c1]
    return image

## Lectura de las series

La tabla muestra qué archivo entra en cada fila y los $G$ obtenidos del `.bval`. Todas las filas deben tener los mismos $G$ para poder alinear las columnas (y restar).

In [ ]:
series = {name: load_series(path) for name, path in ROWS.items()}
stacks = {name: g_stack(s) for name, s in series.items()}

levels = next(iter(stacks.values()))[0]
for name, (g, _) in stacks.items():
    if not np.array_equal(g, levels):
        raise ValueError(f'{name} tiene otros G ({g}) que la primera fila ({levels}).')

grid = {name: images for name, (_, images) in stacks.items()}
if DIFFERENCE is not None:
    a, b = DIFFERENCE
    grid[f'{a} - {b}'] = stacks[a][1] - stacks[b][1]

display(pd.DataFrame([{
    'fila': name, 'archivo': s['path'].name, 'protocolo': s['protocol'], 'volúmenes': s['data'].shape[-1],
    'b_max (s/mm²)': round(float(s['bval'].max()), 2),
} for name, s in series.items()]))
print('G (mT/m):', ', '.join(f'{g:g}' for g in levels))

## Grilla

In [ ]:
signal_names = list(ROWS)
vmax = 1.0 if NORMALIZE else max(np.percentile(stacks[name][1][0], VMAX_PERCENTILE) for name in signal_names)
if DIFFERENCE is not None:
    difference = grid[f'{DIFFERENCE[0]} - {DIFFERENCE[1]}']
    diff_limit = DIFF_LIMIT or float(np.percentile(np.abs(difference), VMAX_PERCENTILE))

n_rows, n_cols = len(grid), len(levels)
height, width = orient(grid[signal_names[0]][0]).shape
fig, axes = plt.subplots(n_rows, n_cols, figsize=(1.45 * n_cols, 1.45 * height / width * n_rows + 0.9),
                         squeeze=False, gridspec_kw={'wspace': 0.03, 'hspace': 0.03})
for i, (name, images) in enumerate(grid.items()):
    is_difference = name not in signal_names
    limits = (-diff_limit, diff_limit) if is_difference else (0.0, vmax)
    for j, g in enumerate(levels):
        ax = axes[i, j]
        ax.imshow(orient(images[j]), cmap='gray', vmin=limits[0], vmax=limits[1], interpolation='nearest')
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        if i == 0:
            ax.set_title(f'{g:g} mT/m', fontsize=9, fontweight='bold')
    axes[i, 0].set_ylabel(name, fontsize=10)

direction_label = 'media de las 3 direcciones' if DIRECTION == 'mean' else f'dirección {DIRECTION}'
fig.suptitle(TITLE or ' | '.join(f'{name}: {s["protocol"]}' for name, s in series.items()) + f' — {direction_label}',
             fontsize=11)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
stem = OUTPUT_NAME or ('__'.join(s['path'].name.split('.nii')[0] for s in series.values())
                       + f'_dir-{DIRECTION}' + ('_norm' if NORMALIZE else ''))
fig.savefig(OUTPUT_DIR / f'{stem}.png', dpi=200, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / f'{stem}.pdf', bbox_inches='tight')
plt.show()
print('Guardado en:', OUTPUT_DIR / f'{stem}.png')